# NYX baryon_density — ParaView visualization data

Saves **Original / SZ3 / NeurLZ (10 ep) / Ours = BO Phase-1+Phase-2 (10 ep)** reconstructions as raw `float32` `512x512x512` volumes at **CR ~300 and ~500**, for ParaView. NeurLZ is single-field; Ours uses the aux fields.

In [1]:
# ── Setup: NYX baryon_density viz-data generator (Original / SZ3 / NeurLZ / Ours) ──
import os, sys, time, random, io, contextlib
import numpy as np, torch
sys.path.append("/home/sam/Halo_Finder/Final_design/base_script")
sys.path.append("/home/sam/Data_Compression/SZ3/tools/pysz")
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from pysz import SZ
from experiment import build_bg_only_cfg
from bg_stage import train_bg_only, run_bg_inference, unwrap_bg_model
from bg_shard import pick_bg_h_under_budget
from monai.networks.nets import BasicUNet

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED = 42
def set_seed(s=SEED):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
sz = SZ("/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so")

NYX_DIR = "/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin/"
SHAPE   = (512, 512, 512)
NYX_ALL = ["baryon_density", "dark_matter_density", "temperature",
           "velocity_x", "velocity_y", "velocity_z"]
TARGET  = "baryon_density"
OUT     = "/storage/sam/Final_visualization"
EPOCHS  = 10
PARAM_BUDGET, BYTES_PER_PARAM = 30000, 2
CR_TARGETS = [300, 500]

def compute_psnr(a, b):
    a = a.astype(np.float64); dr = float(a.max() - a.min()) or 1.0
    mse = float(np.mean((a - b.astype(np.float64)) ** 2))
    return 100.0 if mse <= 0 else 20 * np.log10(dr) - 10 * np.log10(mse)

# slice-direction helpers (data layout Z,Y,X)
DIRECTIONS = ["Z", "Y", "X"]
_FWD = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (2, 0, 1)}
_INV = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (1, 2, 0)}
_AX  = {"Z": 0, "Y": 1, "X": 2}
def permute_fields(fields, d):
    ax = _FWD[d]
    return fields if ax == (0, 1, 2) else [np.ascontiguousarray(np.transpose(f, ax)) for f in fields]
def unpermute(field, d):
    ax = _INV[d]
    return field if ax == (0, 1, 2) else np.ascontiguousarray(np.transpose(field, ax))

# gt  = np.fromfile(NYX_DIR + TARGET + ".f32", np.float32).reshape(SHAPE)
# aux = [np.fromfile(NYX_DIR + a + ".f32", np.float32).reshape(SHAPE) for a in NYX_ALL if a != TARGET]
# _orig = int(gt.nbytes)
# print(f"loaded {TARGET} + {len(aux)} aux | out -> {OUT}")

/home/sam/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── CR search, SZ3 base, and the BG cfg used by our pipeline ──────────────────
def find_rel_for_cr(target_cr, lo=1e-6, hi=5e-4, iters=20):
    """Binary-search rel so the SZ3-only CR ~= target_cr (CR rises with looser rel)."""
    for _ in range(iters):
        mid = (lo * hi) ** 0.5
        b, _ = sz.compress(gt, 1, 0, float(mid), 0)
        if _orig / len(b) < target_cr: lo = mid          # need higher CR -> looser
        else: hi = mid
    rel = (lo * hi) ** 0.5
    b, _ = sz.compress(gt, 1, 0, float(rel), 0)
    return float(rel), _orig / len(b), len(b)

def sz3_recon(rel):
    b, _ = sz.compress(gt, 1, 0, float(rel), 0)
    return np.ascontiguousarray(sz.decompress(b, SHAPE, np.float32), np.float32), len(b)

def build_cfg(Xs_in, Xps_in, bg_h, steps, lr, epochs, patch):
    cfg = build_bg_only_cfg(
        X_target=Xs_in[0], Xps=Xps_in, max_train_time=1e9, epochs=epochs,
        steps_per_epoch=steps, bg_h=bg_h, bg_batch=1, bg_patch_size=patch, lr=lr)
    cfg.bg_sample_mode = "sequential"; cfg.bg_arch = "spatial"
    cfg.amp = False; cfg.amp_dtype = "bf16"; cfg.bg_ddp = False; cfg.bg_data_parallel = False
    cfg.bg_field_norm = "zscore"; cfg.bg_split_mode = "three"; cfg.bg_split_bands = True
    cfg.bg_split_sigma = 0.12; cfg.bg_sigma_low = 0.08; cfg.bg_sigma_mid = 0.18
    cfg.bg_freq_weight = 0.5; cfg.bg_fft_phase_weight = 0.5; cfg.bg_freq_warmup_epochs = 1
    cfg.bg_low_weight = 0.25; cfg.bg_mid_weight = 0.55; cfg.bg_high_weight = 1.10
    cfg.seed = SEED
    return cfg

In [3]:
# ── NeurLZ baseline: single-field BasicUNet on the min-max-normalized SZ3 residual ──
def neurlz_recon(lq, epochs=EPOCHS, lr=1e-2, features=(4, 4, 4, 4, 4, 4), eff=10):
    D, H, W = SHAPE                                        # slice along axis 0 (Z)
    def mm_fit(x, eps=1e-8):
        lo = float(x.min()); hi = float(x.max())
        return ((x - lo) / (hi - lo + eps)).astype(np.float32), (lo, hi)
    def mm_inv(xn, p, eps=1e-8):
        lo, hi = p; return xn * (hi - lo + eps) + lo
    lq_n, _   = mm_fit(lq)
    err_n, ep = mm_fit(gt - lq)
    ph, pw = (-H) % 16, (-W) % 16
    pad = ((0, 0), (0, 0), (0, ph), (0, pw))
    X = torch.from_numpy(np.pad(lq_n[:, None], pad, "reflect"))
    Y = torch.from_numpy(np.pad(err_n[:, None], pad, "reflect"))
    set_seed(SEED)
    with contextlib.redirect_stdout(io.StringIO()):
        m = BasicUNet(spatial_dims=2, features=tuple(features), act="gelu",
                      in_channels=1, out_channels=1).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=1500)
    mse = torch.nn.MSELoss(); idx = np.arange(D); m.train()
    for e in range(epochs):
        np.random.shuffle(idx)
        for st in range(0, D, eff):
            bi = idx[st:st + eff]
            loss = mse(m(X[bi].to(device)), Y[bi].to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step()
    m.eval(); out = lq.copy()
    with torch.no_grad():
        for st in range(0, D, eff):
            bi = list(range(st, min(st + eff, D)))
            pr = m(X[bi].to(device)).cpu().numpy()[:, 0, :H, :W]
            out[bi] = lq[bi] + mm_inv(pr, ep)
    del m
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return np.ascontiguousarray(out, np.float32)

In [4]:
# ── Our pipeline: BO Phase-1 (proxy) picks (lr, dir), Phase-2 trains it `epochs` epochs ──
def ours_recon(rel, epochs=EPOCHS, n_trials=8, tune_depth=32, proxy_target=256, proxy_epochs=3, return_model=False):
    lq, _ = sz3_recon(rel)
    Xs, Xps = [gt] + aux, [lq] + aux

    def make_proxy(fields, d, n_keep=tune_depth, target=proxy_target):
        ax = _AX[d]; fwd = _FWD[d]; n = fields[0].shape[ax]
        H, W = fields[0].shape[fwd[1]], fields[0].shape[fwd[2]]
        sH = max(1, int(round(H / target))); sW = max(1, int(round(W / target)))
        idx = np.unique(np.linspace(0, n - 1, n_keep).round().astype(int))
        out = []
        for f in fields:
            sub = np.transpose(np.take(f, idx, axis=ax), fwd)
            out.append(np.ascontiguousarray(sub[:, ::sH, ::sW]))
        return out

    proxy = {d: (make_proxy(Xs, d), make_proxy(Xps, d)) for d in DIRECTIONS}
    pshape = proxy["Z"][0][0].shape
    try:
        bg_h, _ = pick_bg_h_under_budget(PARAM_BUDGET, shape=pshape, n_fields=len(proxy["Z"][1]), bg_arch="spatial")
        bg_h = int(bg_h)
    except Exception:
        bg_h = 10

    def objective(trial):
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        d  = trial.suggest_categorical("direction", DIRECTIONS)
        Xs_s, Xps_s = proxy[d]; nd = Xs_s[0].shape[0]
        patch = int(min(Xs_s[0].shape[1], Xs_s[0].shape[2]))
        cfg = build_cfg(Xs_s, Xps_s, bg_h, nd, lr, proxy_epochs, patch)
        def ev(m, c=cfg, X=Xs_s, P=Xps_s):
            return compute_psnr(X[0], run_bg_inference(unwrap_bg_model(m), X, P, c, rel)), 0.0
        set_seed(SEED)
        _, h = train_bg_only(Xs=Xs_s, Xps=Xps_s, device=device, cfg=cfg, evaluator=ev)
        ps = [v[1] if isinstance(v, tuple) else v for v in h.get("psnr", [])]
        p = ps[-1] if ps else -1.0
        return p if np.isfinite(p) else -1.0

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4))
    for d in DIRECTIONS:
        study.enqueue_trial({"lr": 1e-3, "direction": d})
    with contextlib.redirect_stdout(io.StringIO()):
        study.optimize(objective, n_trials=n_trials)
    best_lr, best_d = study.best_params["lr"], study.best_params["direction"]
    print(f"  [BO] best lr={best_lr:.2e} dir={best_d} | proxy PSNR={study.best_value:.2f} dB")

    # Phase 2: train the picked config on the full volume for `epochs` epochs
    Xs_p, Xps_p = permute_fields(Xs, best_d), permute_fields(Xps, best_d)
    nd = Xs_p[0].shape[0]; patch = int(Xs_p[0].shape[2])
    try:
        bg_h2, _ = pick_bg_h_under_budget(PARAM_BUDGET, shape=gt.shape, n_fields=len(Xps), bg_arch="spatial")
        bg_h2 = int(bg_h2)
    except Exception:
        bg_h2 = 10
    cfg = build_cfg(Xs_p, Xps_p, bg_h2, nd, best_lr, epochs, patch)
    set_seed(SEED)
    model, _ = train_bg_only(Xs=Xs_p, Xps=Xps_p, device=device, cfg=cfg, evaluator=None)
    core = unwrap_bg_model(model)
    xh = unpermute(run_bg_inference(core, Xs_p, Xps_p, cfg, rel), best_d)
    if return_model:
        # keep the trained model + its (best_d-permuted) tensors so callers can read
        # the 3-band SPLIT HEAD predictions (see the frequency-head band cell below).
        return np.ascontiguousarray(xh, np.float32), best_lr, best_d, core, cfg, Xs_p, Xps_p
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return np.ascontiguousarray(xh, np.float32), best_lr, best_d

In [5]:
# # ── Generate & save all f32 volumes for ParaView (original order, 512^3) ──────
# os.makedirs(OUT, exist_ok=True)
# gt.tofile(OUT + f"{TARGET}_original.f32")
# print(f"saved {TARGET}_original.f32  ({_orig/1e6:.1f} MB)")

# for cr_t in CR_TARGETS:
#     rel, cr, nbytes = find_rel_for_cr(cr_t)
#     tag = f"cr{cr_t}"
#     print(f"\n=== target CR {cr_t}  ->  rel={rel:.2e}, actual SZ3 CR={cr:.1f} (sz {nbytes/1e6:.2f} MB) ===")

#     lq, _ = sz3_recon(rel)
#     lq.tofile(OUT + f"{TARGET}_sz3_{tag}.f32")
#     print(f"  SZ3            PSNR {compute_psnr(gt, lq):6.2f} dB  -> {TARGET}_sz3_{tag}.f32")

#     t0 = time.time()
#     nlz = neurlz_recon(lq, epochs=EPOCHS)
#     nlz.tofile(OUT + f"{TARGET}_neurlz_{tag}.f32")
#     print(f"  NeurLZ({EPOCHS}ep)   PSNR {compute_psnr(gt, nlz):6.2f} dB  -> {TARGET}_neurlz_{tag}.f32  [{time.time()-t0:.0f}s]")

#     t0 = time.time()
#     mine, blr, bd = ours_recon(rel, epochs=EPOCHS)
#     mine.tofile(OUT + f"{TARGET}_ours_{tag}.f32")
#     print(f"  Ours({EPOCHS}ep)     PSNR {compute_psnr(gt, mine):6.2f} dB  -> {TARGET}_ours_{tag}.f32  [{time.time()-t0:.0f}s]")

# print(f"\nDONE. All .f32 (512x512x512, float32, original order) in {OUT}")

In [6]:
# # ── Residual frequency-band volumes for ParaView (512^3, the pipeline's 3-band split) ──
# # The SZ3 residual (gt - SZ3) is exactly what our model learns to predict; training splits
# # it into low / mid / high frequency bands with a per-slice Gaussian FFT split. We export
# # those three bands (plus the full residual) as 512^3 float32 volumes so ParaView can show
# # where the residual's energy lives at each CR. Same split as our cfg (sigma_low / sigma_mid).
# from bg_sampling import _gaussian_low_mid_high_split_t

# RES_SIGMA_LOW, RES_SIGMA_MID = 0.08, 0.18   # == cfg.bg_sigma_low / cfg.bg_sigma_mid

# def residual_bands(resid, sigma_low=RES_SIGMA_LOW, sigma_mid=RES_SIGMA_MID, chunk=64):
#     """Split a (Z,Y,X) residual into (low, mid, high) volumes with the pipeline's
#     per-Z-slice Gaussian FFT split; low + mid + high == resid (masks sum to 1)."""
#     D = resid.shape[0]
#     lo = np.empty_like(resid); mi = np.empty_like(resid); hi = np.empty_like(resid)
#     for s in range(0, D, chunk):
#         x = torch.from_numpy(np.ascontiguousarray(resid[s:s+chunk][:, None])).to(device)  # [b,1,H,W]
#         xl, xm, xh = _gaussian_low_mid_high_split_t(x, sigma_low, sigma_mid)
#         lo[s:s+chunk] = xl[:, 0].cpu().numpy()
#         mi[s:s+chunk] = xm[:, 0].cpu().numpy()
#         hi[s:s+chunk] = xh[:, 0].cpu().numpy()
#     return lo, mi, hi

# os.makedirs(OUT, exist_ok=True)
# for cr_t in CR_TARGETS:
#     rel, cr, nbytes = find_rel_for_cr(cr_t)
#     tag = f"cr{cr_t}"
#     lq, _ = sz3_recon(rel)
#     resid = np.ascontiguousarray(gt - lq, np.float32)          # SZ3 residual = the model's target
#     lo, mi, hi = residual_bands(resid)
#     resid.tofile(OUT + f"{TARGET}_residual_{tag}.f32")         # full residual (reference)
#     lo.tofile(OUT + f"{TARGET}_residual_low_{tag}.f32")
#     mi.tofile(OUT + f"{TARGET}_residual_mid_{tag}.f32")
#     hi.tofile(OUT + f"{TARGET}_residual_high_{tag}.f32")
#     err = float(np.abs(lo + mi + hi - resid).max())
#     print(f"=== CR {cr_t} (rel={rel:.2e}, SZ3 CR={cr:.1f}) | resid range [{resid.min():+.3g}, {resid.max():+.3g}]")
#     print(f"    band std  low={lo.std():.4g}  mid={mi.std():.4g}  high={hi.std():.4g}  | recon err {err:.1e}")
#     print(f"    saved {TARGET}_residual[ |_low|_mid|_high]_{tag}.f32  (512^3 f32)")

# print(f"\nDONE. Residual + 3 frequency-band volumes (512^3 float32) in {OUT}")


In [7]:
# # ── Frequency-HEAD band volumes for ParaView (512^3): the model's own R̂_low/mid/high ──
# # Cell above splits the GROUND-TRUTH SZ3 residual. This cell instead exports what the
# # 3-band SPLIT HEAD of our TRAINED model actually predicts: run the full pipeline (BO +
# # Phase-2), then read the three head outputs (pred_low / pred_mid / pred_high) per Z-slice,
# # denormalize them to physical residual units, and unpermute back to (Z,Y,X). For the
# # pipeline's zscore residual norm the denorm is a scalar (*res_std), so
# # low + mid + high == the model's full residual prediction (additive bands).
# from bg_normalize import (_build_input_norm_tensors, normalize_bg_inputs,
#                           denormalize_bg_residual_tensor, _bg_arch_kind, _bg_residual_norm_mode)

# @torch.no_grad()
# def run_bg_inference_bands(model, Xs, Xps, cfg, rel_err):
#     """Return the model's 3 frequency-head predictions as physical residual volumes.
#     Mirrors run_bg_inference's per-slice spatial path but calls bg_forward_split and
#     denormalizes each head. Returns (low, mid, high, full) each (D,H,W) float32."""
#     if _bg_arch_kind(cfg) == "slab2d":
#         raise NotImplementedError("freq-head band export supports the spatial (slice2d) arch")
#     if _bg_residual_norm_mode(cfg) == "revin_slice":
#         raise NotImplementedError("bands assume a linear (zscore/minmax) residual norm; revin breaks additivity")
#     core = unwrap_bg_model(model); core.eval()
#     dev = next(core.parameters()).device
#     D, H, W = Xps[0].shape
#     mean_t, std_t, min_t, max_t = _build_input_norm_tensors(cfg, dev, len(Xps))
#     lo = np.empty((D, H, W), np.float32); mi = np.empty_like(lo); hi = np.empty_like(lo)
#     zero = torch.zeros(1, device=dev)
#     for z in range(D):
#         st = torch.from_numpy(np.stack([f[z] for f in Xps], 0).astype(np.float32)).unsqueeze(0).to(dev)
#         sn = normalize_bg_inputs(st, cfg, mean_t, std_t, min_t, max_t)
#         zc = torch.tensor([float(z)], device=dev)
#         p_low, p_mid, p_high, _p = core.bg_forward_split(sn, zc, zero, zero, rel_err=rel_err)
#         lo[z] = denormalize_bg_residual_tensor(p_low,  cfg).cpu().numpy()[0, 0]
#         mi[z] = denormalize_bg_residual_tensor(p_mid,  cfg).cpu().numpy()[0, 0]
#         hi[z] = denormalize_bg_residual_tensor(p_high, cfg).cpu().numpy()[0, 0]
#     return lo, mi, hi, (lo + mi + hi)

# def ours_freq_head_bands(rel, epochs=EPOCHS):
#     """Train our pipeline at `rel`, then return the frequency-head 3-band residual
#     volumes in original (Z,Y,X) order: (low, mid, high, full), each 512^3 float32."""
#     _xh, best_lr, best_d, core, cfg, Xs_p, Xps_p = ours_recon(rel, epochs=epochs, return_model=True)
#     lo_p, mi_p, hi_p, full_p = run_bg_inference_bands(core, Xs_p, Xps_p, cfg, rel)
#     out = tuple(np.ascontiguousarray(unpermute(b, best_d), np.float32)
#                 for b in (lo_p, mi_p, hi_p, full_p))
#     del core
#     torch.cuda.empty_cache() if torch.cuda.is_available() else None
#     return out   # (low, mid, high, full)  each 512^3

# # ── Generate, save, and keep the freq-head bands for each CR ──────────────────
# os.makedirs(OUT, exist_ok=True)
# freq_head_bands = {}   # cr -> {"low","mid","high","full"} : each 512^3 volume
# for cr_t in CR_TARGETS:
#     rel, cr, nbytes = find_rel_for_cr(cr_t)
#     tag = f"cr{cr_t}"
#     lo, mi, hi, full = ours_freq_head_bands(rel, epochs=EPOCHS)
#     freq_head_bands[cr_t] = {"low": lo, "mid": mi, "high": hi, "full": full}
#     lo.tofile(  OUT + f"{TARGET}_pred_low_{tag}.f32")
#     mi.tofile(  OUT + f"{TARGET}_pred_mid_{tag}.f32")
#     hi.tofile(  OUT + f"{TARGET}_pred_high_{tag}.f32")
#     full.tofile(OUT + f"{TARGET}_pred_full_{tag}.f32")
#     print(f"=== CR {cr_t} (rel={rel:.2e}, SZ3 CR={cr:.1f}) | freq-head band std: "
#           f"low={lo.std():.4g} mid={mi.std():.4g} high={hi.std():.4g}")
#     print(f"    saved {TARGET}_pred_[low|mid|high|full]_{tag}.f32  (512^3 f32)")

# print(f"\nDONE. Frequency-head 3-band prediction volumes (512^3 f32) in {OUT}")
# print("In-memory: freq_head_bands[CR]['low'|'mid'|'high'|'full'] -> each 512^3 numpy volume.")


In [8]:
# # ── PSNR / RMSE / max-err evaluation of every saved volume vs the original ─────
# # Reads the actual .f32 files back from disk (what ParaView loads) and tabulates
# # per-(method, CR) metrics; also saves psnr_eval.csv.
# import pandas as pd
# _gt = np.fromfile(OUT + f"{TARGET}_original.f32", np.float32).reshape(SHAPE).astype(np.float64)
# _dr = float(_gt.max() - _gt.min()) or 1.0

# def _eval_file(path):
#     x = np.fromfile(path, np.float32).reshape(SHAPE).astype(np.float64)
#     d = _gt - x
#     mse = float(np.mean(d ** 2))
#     return {"PSNR_dB": (100.0 if mse <= 0 else 20*np.log10(_dr) - 10*np.log10(mse)),
#             "RMSE": mse ** 0.5, "max_abs_err": float(np.abs(d).max())}

# rows = []
# for cr_t in CR_TARGETS:
#     tag = f"cr{cr_t}"
#     for method, fn in [("SZ3",    f"{TARGET}_sz3_{tag}.f32"),
#                        ("NeurLZ", f"{TARGET}_neurlz_{tag}.f32"),
#                        ("Ours",   f"{TARGET}_ours_{tag}.f32")]:
#         p = OUT + fn
#         if not os.path.exists(p):
#             print("  [missing]", fn); continue
#         m = _eval_file(p); m.update({"CR": cr_t, "method": method, "file": fn})
#         rows.append(m)

# eval_df = pd.DataFrame(rows)[["CR", "method", "PSNR_dB", "RMSE", "max_abs_err", "file"]]
# print(f"Target: {TARGET}  (original {_gt.size*4/1e6:.1f} MB, range {_dr:.4g})\n")
# print(eval_df.to_string(index=False, formatters={
#     "PSNR_dB":     lambda v: f"{v:7.2f}",
#     "RMSE":        lambda v: f"{v:.4g}",
#     "max_abs_err": lambda v: f"{v:.4g}"}))
# eval_df.to_csv(OUT + "psnr_eval.csv", index=False)
# print(f"\nSaved -> {OUT}psnr_eval.csv")

In [9]:
# # ── SPERR residual normalization study: min-max vs z-score (baryon_density @ CR~500) ──
# # NeurLZ min-max-normalizes the SZ3/SPERR residual to [0,1]. SPERR's residual is
# # heavy-tailed (a few huge wavelet-ringing outliers), so min-max crushes ~all the mass
# # into a sliver around 0.5 -> the MSE objective is misaligned with physical error and
# # NeurLZ can't learn. z-score (mean/std) spreads the bulk across ~[-3,3]. These two
# # histograms show exactly that. (SZ3's residual is L-inf bounded, so it never hits this.)
# import subprocess, os
# import matplotlib.pyplot as plt

# SPERR_BIN = "/home/sam/Halo_Finder/SPERR/build/bin/sperr3d"
# SP_DATA   = NYX_DIR + TARGET + ".f32"          # baryon_density on disk
# CR_TARGET_SP = 500

# def _sperr_cr_probe(data_file, shape, target_psnr):
#     """Compress only -> effective CR (for the target-PSNR search)."""
#     W, H, D = shape[2], shape[1], shape[0]
#     tag = np.random.randint(1 << 30); bit = f"/tmp/vzp_{tag}.bit"
#     subprocess.run([SPERR_BIN, "-c", "--ftype", "32", "--dims", str(W), str(H), str(D),
#                     "--psnr", f"{target_psnr:.4f}", "--bitstream", bit, data_file],
#                    capture_output=True, text=True)
#     n = os.path.getsize(bit); os.remove(bit)
#     return (int(np.prod(shape)) * 4) / n

# def _sperr_recon(data_file, shape, target_psnr):
#     """Compress + decompress -> (recon, CR, nbytes)."""
#     W, H, D = shape[2], shape[1], shape[0]
#     tag = np.random.randint(1 << 30); bit = f"/tmp/vz_{tag}.bit"; dec = f"/tmp/vz_{tag}.dec"
#     subprocess.run([SPERR_BIN, "-c", "--ftype", "32", "--dims", str(W), str(H), str(D),
#                     "--psnr", f"{target_psnr:.4f}", "--bitstream", bit, data_file],
#                    capture_output=True, text=True)
#     nbytes = os.path.getsize(bit)
#     subprocess.run([SPERR_BIN, "-d", "--decomp_f", dec, bit], capture_output=True, text=True)
#     rec = np.fromfile(dec, np.float32).reshape(shape)
#     for x in (bit, dec):
#         if os.path.exists(x): os.remove(x)
#     return rec, (int(np.prod(shape)) * 4) / nbytes, nbytes

# # binary-search SPERR's --psnr target so the effective CR ~= CR_TARGET_SP (CR rises as target falls)
# lo, hi = 1.0, 200.0
# for _ in range(10):
#     mid = (lo + hi) / 2
#     if _sperr_cr_probe(SP_DATA, SHAPE, mid) > CR_TARGET_SP: lo = mid
#     else: hi = mid
# tp = (lo + hi) / 2

# recon_sp, cr_sp, sp_bytes = _sperr_recon(SP_DATA, SHAPE, tp)
# resid = (gt.astype(np.float64) - recon_sp.astype(np.float64)).ravel()
# base_psnr = compute_psnr(gt, recon_sp)
# rstd, rmin, rmax = resid.std(), resid.min(), resid.max()
# print(f"SPERR {TARGET}: target_psnr={tp:.2f} -> CR={cr_sp:.1f}, base PSNR={base_psnr:.2f}")
# print(f"residual: std={rstd:.4g} min={rmin:.4g} max={rmax:.4g} range/std={(rmax-rmin)/rstd:.1f} "
#       f"(99.9%ile |r|={np.percentile(np.abs(resid),99.9):.4g})")

# # the two normalizations NeurLZ could use
# eps = 1e-8
# mm = (resid - rmin) / (rmax - rmin + eps)              # min-max -> [0,1]
# zs = (resid - resid.mean()) / (rstd + eps)             # z-score  (sigma units)

# fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
# axes[0].hist(mm, bins=250, color="tab:red", log=True)
# axes[0].axvline(0.5, color="k", ls=":", lw=1)
# axes[0].set_title(f"min-max  →  [0,1]\nrange/std = {(rmax-rmin)/rstd:.0f}  (outliers own the scale;\nbulk crushed to a sliver at ~0.5)", fontsize=11)
# axes[0].set_xlabel("normalized residual value"); axes[0].set_ylabel("count (log)")
# axes[0].set_xlim(0, 1)

# axes[1].hist(np.clip(zs, -6, 6), bins=250, color="tab:blue", log=True)
# axes[1].set_title("z-score  →  (r − μ)/σ\nbulk spread across ~[−3, 3]  (learnable)", fontsize=11)
# axes[1].set_xlabel("normalized residual (σ, clipped ±6)"); axes[1].set_ylabel("count (log)")
# axes[1].set_xlim(-6, 6)

# for ax in axes: ax.grid(True, alpha=0.3)
# fig.suptitle(f"SPERR residual normalization — {TARGET} @ CR≈{cr_sp:.0f} (SPERR base {base_psnr:.1f} dB)",
#              fontsize=13)
# plt.tight_layout()
# out = f"norm_hist_sperr_{TARGET}_cr{int(round(cr_sp))}.pdf"
# plt.savefig(out, dpi=150, bbox_inches="tight")
# plt.show()
# print("Saved:", out)


In [10]:
# # ── S3D 500^3 visualization data: original volume + SZ3 residual (ParaView) ──
# # Self-contained (doesn't touch the NYX baryon_density globals used above: gt/SHAPE/
# # TARGET/OUT/_orig). S3D is stored as float64 -- converted to float32 here (matches
# # SPERR.py's S3D handling). Saves the original volume once, and for each target CR:
# # the SZ3 reconstruction + the residual (gt - SZ3), all as raw float32 for ParaView.
# S3D_RAW    = "/home/sam/Halo_Finder/halo_finder_v1/s3d-500_500_500-double.raw"
# S3D_SHAPE  = (500, 500, 500)
# S3D_STEM   = "s3d"
# S3D_OUT    = OUT   # same output dir as the NYX volumes above; files are s3d_-prefixed
# S3D_CR_TARGETS = CR_TARGETS   # reuse [300, 500] from cell 1; override here if needed

# s3d_gt = np.fromfile(S3D_RAW, dtype=np.float64).reshape(S3D_SHAPE).astype(np.float32)
# s3d_orig_bytes = int(s3d_gt.nbytes)
# print(f"loaded {S3D_STEM} {S3D_SHAPE} float64->float32 ({s3d_orig_bytes/1e6:.1f} MB) | out -> {S3D_OUT}")

# def s3d_find_rel_for_cr(target_cr, lo=1e-6, hi=5e-3, iters=20):
#     """Binary-search rel so the SZ3-only CR ~= target_cr (CR rises with looser rel)."""
#     for _ in range(iters):
#         mid = (lo * hi) ** 0.5
#         b, _ = sz.compress(s3d_gt, 1, 0, float(mid), 0)
#         if s3d_orig_bytes / len(b) < target_cr: lo = mid       # need higher CR -> looser
#         else: hi = mid
#     rel = (lo * hi) ** 0.5
#     b, _ = sz.compress(s3d_gt, 1, 0, float(rel), 0)
#     return float(rel), s3d_orig_bytes / len(b), len(b)

# def s3d_sz3_recon(rel):
#     b, _ = sz.compress(s3d_gt, 1, 0, float(rel), 0)
#     return np.ascontiguousarray(sz.decompress(b, S3D_SHAPE, np.float32), np.float32), len(b)

# os.makedirs(S3D_OUT, exist_ok=True)
# s3d_gt.tofile(S3D_OUT + f"{S3D_STEM}_original.f32")
# print(f"saved {S3D_STEM}_original.f32  ({s3d_orig_bytes/1e6:.1f} MB)")

# for cr_t in S3D_CR_TARGETS:
#     rel, cr, nbytes = s3d_find_rel_for_cr(cr_t)
#     tag = f"cr{cr_t}"
#     print(f"\n=== {S3D_STEM} target CR {cr_t}  ->  rel={rel:.2e}, actual SZ3 CR={cr:.1f} (sz {nbytes/1e6:.2f} MB) ===")

#     lq, _ = s3d_sz3_recon(rel)
#     lq.tofile(S3D_OUT + f"{S3D_STEM}_sz3_{tag}.f32")
#     print(f"  SZ3       PSNR {compute_psnr(s3d_gt, lq):6.2f} dB  -> {S3D_STEM}_sz3_{tag}.f32")

#     resid = np.ascontiguousarray(s3d_gt - lq, np.float32)
#     resid.tofile(S3D_OUT + f"{S3D_STEM}_residual_{tag}.f32")
#     print(f"  residual  std={resid.std():.4g} range=[{resid.min():+.4g}, {resid.max():+.4g}]"
#           f"  -> {S3D_STEM}_residual_{tag}.f32")

# print(f"\nDONE. {S3D_STEM} original + SZ3 recon + residual (500^3 float32) in {S3D_OUT}")


In [11]:
# # ── Plot: S3D original / SZ3 / residual — middle Z-slice, one row per CR target ──
# # Reads the .f32 files back from disk (what ParaView actually loads), matching the
# # eval cell's philosophy above. One row per CR target: Original | SZ3 | Residual,
# # with a shared color scale for Original/SZ3 (so blur/loss is visually comparable)
# # and its own scale for the residual (very different magnitude).
# import matplotlib.pyplot as plt

# _s3d_gt_disk = np.fromfile(S3D_OUT + f"{S3D_STEM}_original.f32", np.float32).reshape(S3D_SHAPE)
# _z0 = S3D_SHAPE[0] // 2                       # middle Z-slice
# _vmin, _vmax = float(_s3d_gt_disk.min()), float(_s3d_gt_disk.max())

# fig, axes = plt.subplots(len(S3D_CR_TARGETS), 3, figsize=(13, 4.3 * len(S3D_CR_TARGETS)))
# if len(S3D_CR_TARGETS) == 1:
#     axes = axes[None, :]

# for row, cr_t in enumerate(S3D_CR_TARGETS):
#     tag = f"cr{cr_t}"
#     sz3_vol  = np.fromfile(S3D_OUT + f"{S3D_STEM}_sz3_{tag}.f32", np.float32).reshape(S3D_SHAPE)
#     res_vol  = np.fromfile(S3D_OUT + f"{S3D_STEM}_residual_{tag}.f32", np.float32).reshape(S3D_SHAPE)
#     psnr = compute_psnr(_s3d_gt_disk, sz3_vol)
#     res_abs_max = float(np.abs(res_vol[_z0]).max()) or 1.0

#     im0 = axes[row, 0].imshow(_s3d_gt_disk[_z0], cmap="viridis", vmin=_vmin, vmax=_vmax)
#     axes[row, 0].set_title(f"Original (z={_z0})", fontsize=11)
#     im1 = axes[row, 1].imshow(sz3_vol[_z0], cmap="viridis", vmin=_vmin, vmax=_vmax)
#     axes[row, 1].set_title(f"SZ3 @ CR≈{cr_t}  ({psnr:.1f} dB)", fontsize=11)
#     im2 = axes[row, 2].imshow(res_vol[_z0], cmap="seismic", vmin=-res_abs_max, vmax=res_abs_max)
#     axes[row, 2].set_title(f"Residual (gt − SZ3)  std={res_vol.std():.3g}", fontsize=11)

#     for ax in axes[row]:
#         ax.set_xticks([]); ax.set_yticks([])
#     fig.colorbar(im1, ax=axes[row, :2], fraction=0.023, pad=0.01)
#     fig.colorbar(im2, ax=axes[row, 2],  fraction=0.046, pad=0.02)

# fig.suptitle(f"{S3D_STEM.upper()} {S3D_SHAPE} — Original vs SZ3 vs Residual", fontsize=13, y=1.0)
# plt.tight_layout()
# _out_png = S3D_OUT + f"{S3D_STEM}_original_sz3_residual.png"
# plt.savefig(_out_png, dpi=150, bbox_inches="tight")
# plt.show()
# print("Saved:", _out_png)


In [12]:
# ── Aligned-CR viz: SPERR + parameterized Ours/NeurLZ helpers (Miranda & Temperature) ──
# Adds SPERR support + dataset-agnostic versions of the Ours pipeline and the NeurLZ
# baseline, plus a CR-ALIGNMENT driver: SPERR+Ours is built first and its effective
# compression ratio becomes the reference; every other compressor / pipeline is then
# tuned (byte-budget binary search) so its OWN effective CR matches that reference.
# Requires cells 1-2 to have run (uses sz, build_cfg, DIRECTIONS/_FWD/_AX, permute_fields,
# unpermute, set_seed, SEED, device, BYTES_PER_PARAM, and the base_script imports).
import subprocess, glob
from experiment import estimate_bg_model_param_bytes

NLZ_FEATURES = (4, 4, 4, 4, 4, 4)   # matches SPERR.py's NeurLZ (fixed-width BasicUNet)
SPERR_BIN    = "/home/sam/Halo_Finder/SPERR/build/bin/sperr3d"


def _sperr_env():
    """LD_LIBRARY_PATH extended to wherever libSPERR.so* lives (avoids rc=127 on
    machines where the .so isn't on the default loader path)."""
    env = os.environ.copy(); root = os.path.dirname(SPERR_BIN); found = set()
    for _ in range(4):
        for hit in glob.glob(os.path.join(root, "**", "libSPERR.so*"), recursive=True):
            found.add(os.path.dirname(hit))
        root = os.path.dirname(root)
        if not root or root == "/":
            break
    if found:
        env["LD_LIBRARY_PATH"] = ":".join(found) + ((":" + env["LD_LIBRARY_PATH"])
                                                     if env.get("LD_LIBRARY_PATH") else "")
    return env


_SPERR_ENV = _sperr_env()


def run_sperr(data_file, target_gt, shape, drange, target_psnr):
    """SPERR at a target PSNR. Returns (CR, PSNR, recon, nbytes). pid+ns tmp names."""
    W, H, D = shape[2], shape[1], shape[0]
    tag = f"{os.getpid()}_{time.time_ns()}"
    bit = f"/tmp/sperr_{tag}.bit"; rec = f"/tmp/sperr_{tag}.dec.f32"
    subprocess.run([SPERR_BIN, "-c", "--ftype", "32", "--dims", str(W), str(H), str(D),
                    "--psnr", f"{float(target_psnr):.4f}", "--bitstream", bit, data_file],
                   capture_output=True, text=True, env=_SPERR_ENV)
    if not os.path.exists(bit):
        return None, None, None, None
    nbytes = os.path.getsize(bit)
    subprocess.run([SPERR_BIN, "-d", "--decomp_f", rec, bit],
                   capture_output=True, text=True, env=_SPERR_ENV)
    cr = psnr = recon = None
    if os.path.exists(rec):
        recon = np.fromfile(rec, dtype=np.float32).reshape(shape)
        a = np.asarray(target_gt, np.float64)
        mse = float(np.mean((a - recon.astype(np.float64)) ** 2))
        psnr = 100.0 if mse <= 0 else 20 * np.log10(drange) - 10 * np.log10(mse)
        cr = (int(np.prod(shape)) * 4) / nbytes
    for f in (bit, rec):
        if os.path.exists(f):
            os.remove(f)
    return cr, psnr, recon, nbytes


def sperr_for_target_bytes(data_file, gt_vol, shape, drange, target_bytes,
                           lo=1.0, hi=250.0, iters=12, label="SPERR"):
    """Binary-search SPERR --psnr so the bitstream size ~= target_bytes; returns
    (recon, nbytes, psnr, cr). Each pass is a FULL compress+decompress of the volume,
    so on 1024^3 this is the slow part -- per-pass progress is printed so you can see
    it working (and spot a silent SPERR failure, which prints 'FAILED')."""
    orig = int(np.prod(shape)) * 4
    target_cr = orig / float(target_bytes)
    print(f"      [{label}] byte-search -> {target_bytes/1e6:.2f} MB (CR {target_cr:.0f}) | up to {iters} passes")
    for it in range(iters):
        mid = 0.5 * (lo + hi); t = time.time()
        cr, ps, _, nb = run_sperr(data_file, gt_vol, shape, drange, mid)
        if cr is None:
            print(f"        {it+1:2d}/{iters}  psnr={mid:6.1f} -> FAILED ({time.time()-t:.0f}s)")
            lo = mid; continue
        print(f"        {it+1:2d}/{iters}  psnr={mid:6.1f} -> {nb/1e6:6.2f} MB  CR {cr:6.1f}  ({time.time()-t:.0f}s)")
        if cr > target_cr:   # too few bytes -> raise quality
            lo = mid
        else:
            hi = mid
    cr, psnr, recon, nbytes = run_sperr(data_file, gt_vol, shape, drange, 0.5 * (lo + hi))
    print(f"      [{label}] picked {nbytes/1e6:.2f} MB @ {psnr:.1f} dB (CR {cr:.1f})")
    return recon, nbytes, psnr, cr


def sz3_for_target_bytes(gt_vol, shape, target_bytes, lo=1e-7, hi=2e-1, iters=16, label="SZ3"):
    """Binary-search SZ3 rel so the bitstream size ~= target_bytes; returns
    (recon, nbytes, rel). Each pass is a full SZ3 compress of the volume; per-pass
    progress is printed."""
    orig = int(np.prod(shape)) * 4
    print(f"      [{label}] byte-search -> {target_bytes/1e6:.2f} MB (CR {orig/target_bytes:.0f}) | {iters} passes")
    for it in range(iters):
        mid = (lo * hi) ** 0.5; t = time.time()
        b, _ = sz.compress(gt_vol, 1, 0, float(mid), 0); nb = len(b)
        print(f"        {it+1:2d}/{iters}  rel={mid:.2e} -> {nb/1e6:6.2f} MB  CR {orig/nb:6.1f}  ({time.time()-t:.0f}s)")
        if nb > target_bytes:   # too many bytes -> loosen (bigger rel)
            lo = mid
        else:
            hi = mid
    rel = (lo * hi) ** 0.5
    b, _ = sz.compress(gt_vol, 1, 0, float(rel), 0)
    recon = np.ascontiguousarray(sz.decompress(b, shape, np.float32), np.float32)
    print(f"      [{label}] picked {len(b)/1e6:.2f} MB (CR {orig/len(b):.1f}) rel={rel:.2e}")
    return recon, len(b), float(rel)


def _basicunet_nparams(features, n_fields):
    with contextlib.redirect_stdout(io.StringIO()):
        m = BasicUNet(spatial_dims=2, features=tuple(features), act="gelu",
                      in_channels=int(n_fields), out_channels=1)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad); del m
    return int(n)


def neurlz_recon_base(gt_vol, base_recon, aux_list, shape, epochs,
                      features=NLZ_FEATURES, lr=1e-2, eff_batch=10):
    """SZ3/SPERR + NeurLZ on an arbitrary base reconstruction (multi-field, min-max,
    MSE, cosine) -- matches SPERR.py's run_neurlz recipe. Returns (recon, n_params)."""
    D, H, W = shape
    def mm(x, eps=1e-8):
        lo, hi = float(x.min()), float(x.max())
        return ((x - lo) / (hi - lo + eps)).astype(np.float32), (lo, hi)
    lq = np.ascontiguousarray(base_recon, np.float32)
    fields = [lq] + [np.asarray(a, np.float32) for a in aux_list]
    n_fields = len(fields)
    lq_n = np.stack([mm(f)[0] for f in fields], axis=1)
    err_n, (e_lo, e_hi) = mm(np.asarray(gt_vol, np.float32) - lq)
    ph, pw = (-H) % 16, (-W) % 16
    pad = ((0, 0), (0, 0), (0, ph), (0, pw))
    X = torch.from_numpy(np.pad(lq_n, pad, "reflect"))
    Y = torch.from_numpy(np.pad(err_n[:, None], pad, "reflect"))
    set_seed(SEED)
    with contextlib.redirect_stdout(io.StringIO()):
        m = BasicUNet(spatial_dims=2, features=tuple(features), act="gelu",
                      in_channels=n_fields, out_channels=1).to(device)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=1500)
    mse = torch.nn.MSELoss(); idx = np.arange(D); m.train()
    for e in range(int(epochs)):
        np.random.shuffle(idx)
        for st in range(0, D, eff_batch):
            bi = idx[st:st + eff_batch]
            loss = mse(m(X[bi].to(device)), Y[bi].to(device))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step()
    m.eval(); out = lq.copy()
    with torch.no_grad():
        for st in range(0, D, eff_batch):
            bi = list(range(st, min(st + eff_batch, D)))
            pr = m(X[bi].to(device)).cpu().numpy()[:, 0, :H, :W]
            out[bi] = lq[bi] + (pr * (e_hi - e_lo + 1e-8) + e_lo)
    del m, X, Y
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return np.ascontiguousarray(out, np.float32), int(n_params)


def ours_recon_base(gt_vol, aux_list, shape, base_recon, base_rel, budget, epochs,
                    n_trials=8, tune_depth=32, proxy_target=256, proxy_epochs=3):
    """Our BG pipeline on an arbitrary base reconstruction: BO Phase-1 (proxy) picks
    (lr, dir), Phase-2 trains `epochs` on the full volume. Returns (recon, nn_bytes, n_params)."""
    Xs  = [np.asarray(gt_vol, np.float32)] + list(aux_list)
    Xps = [np.ascontiguousarray(base_recon, np.float32)] + list(aux_list)

    def _psnr(a, b):
        a = a.astype(np.float64); dr = float(a.max() - a.min()) or 1.0
        mse = float(np.mean((a - b.astype(np.float64)) ** 2))
        return 100.0 if mse <= 0 else 20 * np.log10(dr) - 10 * np.log10(mse)

    def make_proxy(fields, d, n_keep=tune_depth, target=proxy_target):
        ax = _AX[d]; fwd = _FWD[d]; n = fields[0].shape[ax]
        H, W = fields[0].shape[fwd[1]], fields[0].shape[fwd[2]]
        sH = max(1, int(round(H / target))); sW = max(1, int(round(W / target)))
        idx = np.unique(np.linspace(0, n - 1, n_keep).round().astype(int))
        out = []
        for f in fields:
            sub = np.transpose(np.take(f, idx, axis=ax), fwd)
            out.append(np.ascontiguousarray(sub[:, ::sH, ::sW]))
        return out

    proxy = {d: (make_proxy(Xs, d), make_proxy(Xps, d)) for d in DIRECTIONS}
    pshape = proxy["Z"][0][0].shape
    try:
        bg_h = int(pick_bg_h_under_budget(budget, shape=pshape, n_fields=len(proxy["Z"][1]),
                                          bg_arch="spatial", h_candidates=list(range(3, 256)))[0])
    except Exception:
        bg_h = 10

    def objective(trial):
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        d  = trial.suggest_categorical("direction", DIRECTIONS)
        Xs_s, Xps_s = proxy[d]; nd = Xs_s[0].shape[0]
        patch = int(min(Xs_s[0].shape[1], Xs_s[0].shape[2]))
        cfg = build_cfg(Xs_s, Xps_s, bg_h, nd, lr, proxy_epochs, patch)
        cfg.bg_cr_rel_err = float(base_rel)
        def ev(m, c=cfg, X=Xs_s, P=Xps_s):
            return _psnr(X[0], run_bg_inference(unwrap_bg_model(m), X, P, c, float(base_rel))), 0.0
        set_seed(SEED)
        with contextlib.redirect_stdout(io.StringIO()):
            _, h = train_bg_only(Xs=Xs_s, Xps=Xps_s, device=device, cfg=cfg, evaluator=ev)
        ps = [v[1] if isinstance(v, tuple) else v for v in h.get("psnr", [])]
        p = ps[-1] if ps else -1.0
        return p if np.isfinite(p) else -1.0

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4))
    for d in DIRECTIONS:
        study.enqueue_trial({"lr": 1e-3, "direction": d})
    with contextlib.redirect_stdout(io.StringIO()):
        study.optimize(objective, n_trials=n_trials)
    best_lr, best_d = study.best_params["lr"], study.best_params["direction"]
    print(f"      [BO] lr={best_lr:.2e} dir={best_d} | proxy PSNR={study.best_value:.2f} dB")

    Xs_p, Xps_p = permute_fields(Xs, best_d), permute_fields(Xps, best_d)
    nd = Xs_p[0].shape[0]; patch = int(Xs_p[0].shape[2])
    try:
        bg_h2 = int(pick_bg_h_under_budget(budget, shape=shape, n_fields=len(Xps),
                                           bg_arch="spatial", h_candidates=list(range(3, 256)))[0])
    except Exception:
        bg_h2 = 10
    cfg = build_cfg(Xs_p, Xps_p, bg_h2, nd, best_lr, epochs, patch)
    cfg.bg_cr_rel_err = float(base_rel)
    set_seed(SEED)
    model, _ = train_bg_only(Xs=Xs_p, Xps=Xps_p, device=device, cfg=cfg, evaluator=None)
    core = unwrap_bg_model(model)
    xh = unpermute(run_bg_inference(core, Xs_p, Xps_p, cfg, float(base_rel)), best_d)
    n_params, nn_bytes = estimate_bg_model_param_bytes(
        n_fields=len(Xps), shape=shape, bg_arch="spatial", bg_h=bg_h2, dtype_bytes=BYTES_PER_PARAM)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return np.ascontiguousarray(xh, np.float32), int(nn_bytes), int(n_params)


def align_and_save(dataset, gt_vol, aux_list, shape, data_file, budget, epochs, target_cr, out_dir):
    """Build all 7 outputs (original + SZ3/SPERR x {solo,+Ours,+NeurLZ}) at ONE aligned
    effective CR, and save each recon + its residual (original - recon) as .f32.
    Alignment: SPERR+Ours is built first; its realized effective CR becomes the
    reference, and every other method's compressed-base byte budget is chosen so its
    own effective CR matches that reference."""
    os.makedirs(out_dir, exist_ok=True)
    gt_vol = np.ascontiguousarray(gt_vol, np.float32)
    orig = int(np.prod(shape)) * 4
    dr = float(gt_vol.max() - gt_vol.min()) or 1.0
    n_fields = 1 + len(aux_list)
    bg_h = int(pick_bg_h_under_budget(budget, shape=shape, n_fields=n_fields,
                                      bg_arch="spatial", h_candidates=list(range(3, 256)))[0])
    _, nn_bytes = estimate_bg_model_param_bytes(
        n_fields=n_fields, shape=shape, bg_arch="spatial", bg_h=bg_h, dtype_bytes=BYTES_PER_PARAM)
    nlz_bytes = _basicunet_nparams(NLZ_FEATURES, n_fields) * BYTES_PER_PARAM

    def _psnr(x):
        a = gt_vol.astype(np.float64)
        mse = float(np.mean((a - x.astype(np.float64)) ** 2))
        return 100.0 if mse <= 0 else 20 * np.log10(dr) - 10 * np.log10(mse)

    def _save(name, recon, eff_cr):
        recon = np.ascontiguousarray(recon, np.float32)
        recon.tofile(os.path.join(out_dir, f"{dataset}_{name}.f32"))
        (gt_vol - recon).astype(np.float32).tofile(os.path.join(out_dir, f"{dataset}_{name}_residual.f32"))
        print(f"    saved {dataset}_{name}.f32 (+_residual)  PSNR {_psnr(recon):6.2f} dB  eff-CR {eff_cr:6.1f}")

    _t0 = time.time()
    def _step(msg):
        print(f"\n  [{dataset}] >>> {msg}   (+{time.time()-_t0:.0f}s elapsed)")

    print(f"\n{'='*70}\n[{dataset}] shape={shape}  budget={budget:,}  epochs={epochs}  target CR~{target_cr}")
    print(f"          bg_h={bg_h}  model={nn_bytes/1e3:.1f}KB  NeurLZ={nlz_bytes/1e3:.1f}KB  n_fields={n_fields}")
    print(f"          NOTE: each compressor pass below is a full compress of a {orig/1e6:.0f} MB volume; "
          f"1024^3 passes take a while (progress printed per pass).")

    gt_vol.tofile(os.path.join(out_dir, f"{dataset}_original.f32"))
    print(f"    saved {dataset}_original.f32  ({orig/1e6:.1f} MB)")

    # ---- 1) SPERR + Ours FIRST -> defines the reference effective CR (cr_star) ----
    _step("STEP 1/7  SPERR+Ours (anchor): SPERR base byte-search")
    b_pipe0 = orig / float(target_cr) - nn_bytes
    rec_sp, nb_sp, ps_sp, _ = sperr_for_target_bytes(data_file, gt_vol, shape, dr, b_pipe0, label="SPERR+Ours base")
    cr_star = orig / (nb_sp + nn_bytes)                         # <-- SPERR+Ours effective CR
    rel_sp  = float(np.abs(gt_vol - rec_sp).max()) / dr
    print(f"      training Ours on the SPERR base ({epochs} epochs) ...")
    enh_sp, _, _ = ours_recon_base(gt_vol, aux_list, shape, rec_sp, rel_sp, budget, epochs)
    print(f"  >> SPERR+Ours reference effective CR = {cr_star:.2f}  "
          f"(SPERR base {nb_sp/1e6:.3f} MB @ {ps_sp:.1f} dB + model {nn_bytes/1e3:.1f} KB)")
    _save("sperr_ours", enh_sp, cr_star); del enh_sp

    # ---- byte budgets so every OTHER method lands on cr_star ----
    b_solo = orig / cr_star                    # standalone compressor
    b_pipe = orig / cr_star - nn_bytes         # X + Ours   (base + model)
    b_nlz  = orig / cr_star - nlz_bytes        # X + NeurLZ (base + NeurLZ)

    # ---- 2) SPERR standalone + SPERR + NeurLZ ----
    _step("STEP 2/7  SPERR (standalone): byte-search")
    rec_sp_solo, nb, _, _ = sperr_for_target_bytes(data_file, gt_vol, shape, dr, b_solo, label="SPERR")
    _save("sperr", rec_sp_solo, orig / nb); del rec_sp_solo
    _step("STEP 3/7  SPERR+NeurLZ: SPERR base byte-search + NeurLZ train")
    rec_sp_nlz, nb_spn, _, _ = sperr_for_target_bytes(data_file, gt_vol, shape, dr, b_nlz, label="SPERR+NeurLZ base")
    print(f"      training NeurLZ on the SPERR base ({epochs} epochs) ...")
    enh_spn, _ = neurlz_recon_base(gt_vol, rec_sp_nlz, aux_list, shape, epochs)
    _save("sperr_neurlz", enh_spn, orig / (nb_spn + nlz_bytes)); del rec_sp_nlz, enh_spn

    del rec_sp   # anchor SPERR base already consumed by SPERR+Ours above
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # ---- 3) SZ3 standalone + SZ3 + Ours + SZ3 + NeurLZ ----
    _step("STEP 4/7  SZ3 (standalone): byte-search")
    rec_sz_solo, nb, _ = sz3_for_target_bytes(gt_vol, shape, b_solo, label="SZ3")
    _save("sz3", rec_sz_solo, orig / nb); del rec_sz_solo
    _step("STEP 5/7  SZ3+Ours: SZ3 base byte-search + Ours train")
    rec_sz_pipe, nb_szp, rel_szp = sz3_for_target_bytes(gt_vol, shape, b_pipe, label="SZ3+Ours base")
    print(f"      training Ours on the SZ3 base ({epochs} epochs) ...")
    enh_szp, _, _ = ours_recon_base(gt_vol, aux_list, shape, rec_sz_pipe, rel_szp, budget, epochs)
    _save("sz3_ours", enh_szp, orig / (nb_szp + nn_bytes)); del rec_sz_pipe, enh_szp
    _step("STEP 6/7  SZ3+NeurLZ: SZ3 base byte-search + NeurLZ train")
    rec_sz_nlz, nb_szn, rel_szn = sz3_for_target_bytes(gt_vol, shape, b_nlz, label="SZ3+NeurLZ base")
    print(f"      training NeurLZ on the SZ3 base ({epochs} epochs) ...")
    enh_szn, _ = neurlz_recon_base(gt_vol, rec_sz_nlz, aux_list, shape, epochs)
    _save("sz3_neurlz", enh_szn, orig / (nb_szn + nlz_bytes)); del rec_sz_nlz, enh_szn
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"\n[{dataset}] DONE in {time.time()-_t0:.0f}s -> {out_dir}")


print("aligned-CR viz helpers ready (run_sperr / sz3_for_target_bytes / ours_recon_base / "
      "neurlz_recon_base / align_and_save)")


aligned-CR viz helpers ready (run_sperr / sz3_for_target_bytes / ours_recon_base / neurlz_recon_base / align_and_save)


In [ ]:
# ── Miranda 1024³ — aligned-CR viz data (SPERR/SZ3 x {solo,+Ours,+NeurLZ} + original) ──
# Saves each recon + its residual to VIZ_OUT. SPERR+Ours defines the reference CR; the
# rest are byte-budget-matched to it. Miranda has no aux -> single-field NeurLZ/Ours.
VIZ_OUT       = "/storage/sam/Final_visualization/miranda_temperature_aligned/"
MIR_FILE      = "/home/sam/Halo_Finder/halo_finder_v1/miranda_1024x1024x1024_float32.raw"
MIR_SHAPE     = (1024, 1024, 1024)
MIR_BUDGET    = 240_000          # same param budget as SPERR.py's Miranda
MIR_EPOCHS    = 5                # <-- Miranda: 5 epochs
MIR_TARGET_CR = 300             # aligned effective CR (edit to taste; realized CR is printed)

set_seed(SEED)
_mir = np.fromfile(MIR_FILE, dtype=np.float32).reshape(MIR_SHAPE)
align_and_save("miranda", _mir, [], MIR_SHAPE, MIR_FILE,
               MIR_BUDGET, MIR_EPOCHS, MIR_TARGET_CR, VIZ_OUT)
del _mir
torch.cuda.empty_cache() if torch.cuda.is_available() else None



[Model: spatial] Total Params: 724
 [Params] Main (BG) Network : 724 parameters

[Model: spatial] Total Params: 1,216
 [Params] Main (BG) Network : 1,216 parameters

[Model: spatial] Total Params: 1,834
 [Params] Main (BG) Network : 1,834 parameters

[Model: spatial] Total Params: 2,578
 [Params] Main (BG) Network : 2,578 parameters

[Model: spatial] Total Params: 3,448
 [Params] Main (BG) Network : 3,448 parameters

[Model: spatial] Total Params: 4,444
 [Params] Main (BG) Network : 4,444 parameters

[Model: spatial] Total Params: 5,566
 [Params] Main (BG) Network : 5,566 parameters

[Model: spatial] Total Params: 6,814
 [Params] Main (BG) Network : 6,814 parameters

[Model: spatial] Total Params: 8,188
 [Params] Main (BG) Network : 8,188 parameters

[Model: spatial] Total Params: 9,688
 [Params] Main (BG) Network : 9,688 parameters

[Model: spatial] Total Params: 11,314
 [Params] Main (BG) Network : 11,314 parameters

[Model: spatial] Total Params: 13,066
 [Params] Main (BG) Network 

/home/sam/Halo_Finder/Final_design/base_script/bg_stage.py:544: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp, dtype=autocast_dtype):


      [BO] lr=8.69e-03 dir=Z | proxy PSNR=50.04 dB

[Model: spatial] Total Params: 724
 [Params] Main (BG) Network : 724 parameters

[Model: spatial] Total Params: 1,216
 [Params] Main (BG) Network : 1,216 parameters

[Model: spatial] Total Params: 1,834
 [Params] Main (BG) Network : 1,834 parameters

[Model: spatial] Total Params: 2,578
 [Params] Main (BG) Network : 2,578 parameters

[Model: spatial] Total Params: 3,448
 [Params] Main (BG) Network : 3,448 parameters

[Model: spatial] Total Params: 4,444
 [Params] Main (BG) Network : 4,444 parameters

[Model: spatial] Total Params: 5,566
 [Params] Main (BG) Network : 5,566 parameters

[Model: spatial] Total Params: 6,814
 [Params] Main (BG) Network : 6,814 parameters

[Model: spatial] Total Params: 8,188
 [Params] Main (BG) Network : 8,188 parameters

[Model: spatial] Total Params: 9,688
 [Params] Main (BG) Network : 9,688 parameters

[Model: spatial] Total Params: 11,314
 [Params] Main (BG) Network : 11,314 parameters

[Model: spatial

In [ ]:
# ── NYX temperature 512³ — aligned-CR viz data (SPERR/SZ3 x {solo,+Ours,+NeurLZ} + original) ──
# Same procedure as the Miranda cell; temperature uses the other NYX fields as aux
# (charged-free side info), so NeurLZ/Ours are multi-field here.
TEMP_TARGET    = "temperature"
TEMP_SHAPE     = (512, 512, 512)
TEMP_BUDGET    = 30_000          # same param budget as SPERR.py's NYX
TEMP_EPOCHS    = 10              # <-- temperature: 10 epochs
TEMP_TARGET_CR = 300            # aligned effective CR (edit to taste; realized CR is printed)

set_seed(SEED)
_tgt = np.fromfile(NYX_DIR + TEMP_TARGET + ".f32", np.float32).reshape(TEMP_SHAPE)
_aux = [np.fromfile(NYX_DIR + a + ".f32", np.float32).reshape(TEMP_SHAPE)
        for a in NYX_ALL if a != TEMP_TARGET]
align_and_save("temperature", _tgt, _aux, TEMP_SHAPE,
               NYX_DIR + TEMP_TARGET + ".f32",
               TEMP_BUDGET, TEMP_EPOCHS, TEMP_TARGET_CR, VIZ_OUT)
del _tgt, _aux
torch.cuda.empty_cache() if torch.cuda.is_available() else None
